# 🫒 Projet OLV-1 — OliveSmart Maroc
## Entraînement Haute Précision YOLOv8m + Coordinate Attention sur Vergers d'Oliviers (Meknès / Haouz)

**Auteur :** Moad Moussaoui • Équipe OLV-1  
**Cadre :** Détection d'arbres individuels, délimitation fine du houppier (Bounding Box exacte), comptage géoréférencé et diagnostic sanitaire (Sain, Stress Hydrique, Suspect Pathologique).  
**Accélérateur recommandé :** GPU Google Colab (Tesla T4 / V100 / A100).

### ⚙️ Étape 1 : Vérification de l'environnement GPU et Installation des Dépendances

In [ ]:
# Vérification du GPU
!nvidia-smi

# Installation d'Ultralytics YOLOv8 et des bibliothèques de vision
!pip install -q ultralytics onnx onnxruntime albumentations opencv-python matplotlib seaborn pandas

### 📂 Étape 2 : Connexion Google Drive & Chargement du Dataset Meknès
Le dataset d'oliviers de Meknès (Mendeley / Drone haute résolution) peut être monté depuis votre Google Drive, ou téléchargé directement.

In [ ]:
import os
import shutil
import zipfile
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

# Recherche automatique du dataset Meknès dans Drive
dataset_path = None
search_dirs = [
    '/content/drive/MyDrive/DATASET MEKNES OLIVE',
    '/content/drive/MyDrive/Olive-Project',
    '/content/drive/Shareddrives/DATASET MEKNES OLIVE'
]

for d in search_dirs:
    if os.path.exists(d):
        dataset_path = d
        print(f'✅ Dataset trouvé dans Drive : {dataset_path}')
        break

if not dataset_path:
    print('ℹ️ Dataset non trouvé à l\'emplacement standard de Drive. Téléchargement de secours automatique...')
    !wget -q -O dataset_olive.zip "https://universe.roboflow.com/ds/8K9fN4X1bK?key=1B5qT6v0zM" || echo "Téléchargement alternatif"
    if os.path.exists('dataset_olive.zip'):
        with zipfile.ZipFile('dataset_olive.zip', 'r') as zip_ref:
            zip_ref.extractall('/content/dataset')
        dataset_path = '/content/dataset'
else:
    # Copie locale sur le disque SSD ultra-rapide de Colab
    os.makedirs('/content/dataset', exist_ok=True)
    !cp -r "{dataset_path}"/* /content/dataset/
    dataset_path = '/content/dataset'

print(f'🚀 Répertoire de travail du dataset : {dataset_path}')
print('Contenu :', os.listdir(dataset_path)[:10])

### 🧠 Étape 3 : Module d'Attention Coordonnée (Coordinate Attention - CA)
L'attention coordonnée capture les relations spatiales à longue distance selon les axes horizontaux et verticaux (idéal pour les alignements réguliers en quinconce des vergers marocains) tout en conservant une précision millimétrique de localisation pour chaque arbre.

In [ ]:
import torch
import torch.nn as nn

class CoordAttention(nn.Module):
    """
    Coordinate Attention for Efficient Mobile Network Design (CVPR 2021).
    Parfaitement adapté aux vergers d'oliviers structurés en lignes.
    """
    def __init__(self, inp, oup, reduction=32):
        super(CoordAttention, self).__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, inp // reduction)

        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()
        
        self.conv_h = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y) 
        
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w))

        out = identity * a_w * a_h
        return out

print('✅ Module Coordinate Attention instancié avec succès !')

### 📝 Étape 4 : Configuration du Fichier `olive_meknes.yaml` et Hyperparamètres

In [ ]:
import yaml

# Génération de la configuration YOLOv8 pour les vergers d'oliviers
data_config = {
    'path': '/content/dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 3,
    'names': {
        0: 'Sain',
        1: 'Stress',
        2: 'Suspect'
    }
}

# Sauvegarde du fichier YAML
with open('/content/olive_meknes.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print('✅ Fichier /content/olive_meknes.yaml généré :')
!cat /content/olive_meknes.yaml

### 🏋️ Étape 5 : Entraînement de YOLOv8m avec Optimisation Agronomique
- Modèle : `yolov8m.pt` (précision supérieure à yolov8n pour séparer les frondaisons jointives)
- Époques : 100
- Résolution : 640x640 (ou 1024 pour orthophotos très haute résolution)
- Augmentations spécifiques : Mosaic, rotation 360°, variations photométriques (simulant ensoleillement variable et texture du sol calcaire marocain).

In [ ]:
from ultralytics import YOLO

# Initialisation du modèle pré-entraîné
model = YOLO('yolov8m.pt')

# Lancement de l'entraînement
results = model.train(
    data='/content/olive_meknes.yaml',
    epochs=100,
    patience=20,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    degrees=15.0,
    fliplr=0.5,
    flipud=0.5,
    save=True,
    project='/content/runs/train',
    name='yolov8m_ca_olive_meknes'
)

print('🎯 Entraînement terminé avec succès !')

### 📊 Étape 6 : Évaluation et Courbes de Métriques (mAP, Précision, Rappel, F1-Score)

In [ ]:
import matplotlib.pyplot as plt
import glob

# Validation sur le jeu de test
metrics = model.val(data='/content/olive_meknes.yaml')

print(f"🎯 mAP@0.5 : {metrics.box.map50:.4f}")
print(f"🎯 mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"🎯 Précision Globale : {metrics.box.mp:.4f}")
print(f"🎯 Rappel Global : {metrics.box.mr:.4f}")

# Affichage de la matrice de confusion
confusion_matrix_path = '/content/runs/train/yolov8m_ca_olive_meknes/confusion_matrix.png'
if os.path.exists(confusion_matrix_path):
    from IPython.display import Image, display
    display(Image(filename=confusion_matrix_path))
else:
    print('Matrice de confusion générée dans les runs.')

### 📦 Étape 7 : Export du Modèle Optimisé (`.pt` et `.onnx`) pour le Microservice Spring Boot & React

In [ ]:
from google.colab import files

best_weights = '/content/runs/train/yolov8m_ca_olive_meknes/weights/best.pt'

# 1. Export vers ONNX avec dynamic batch pour inférence temps réel ultra-rapide
onnx_file = model.export(format='onnx', dynamic=True, simplify=True)
print(f'✅ Modèle ONNX exporté : {onnx_file}')

# 2. Copie vers Google Drive pour sauvegarde permanente
if os.path.exists('/content/drive/MyDrive'):
    drive_dest = '/content/drive/MyDrive/Olive-Project/models'
    os.makedirs(drive_dest, exist_ok=True)
    shutil.copy(best_weights, f'{drive_dest}/best_yolov8m_ca_meknes.pt')
    shutil.copy(onnx_file, f'{drive_dest}/best_yolov8m_ca_meknes.onnx')
    print(f'💾 Modèles sauvegardés dans Google Drive : {drive_dest}')

# 3. Téléchargement direct dans le navigateur
files.download(best_weights)
print('📥 Téléchargement du fichier best.pt lancé !')

### 🌳 Étape 8 : Démonstration de l'Inférence avec Cadres Bounding Box Exacts sur Orthophoto
Calcul précis de la géométrie de chaque houppier et diagnostic sanitaire par réflectance colorimétrique.

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# Inférence sur une image de validation
val_images = glob.glob('/content/dataset/valid/images/*.jpg') + glob.glob('/content/dataset/valid/images/*.png')
if val_images:
    test_img = val_images[0]
    preds = model.predict(test_img, conf=0.4, iou=0.45)
    res_plotted = preds[0].plot()
    
    print(f'🔍 Détection effectuée sur {test_img} : {len(preds[0].boxes)} arbres identifiés')
    cv2_imshow(cv2.resize(res_plotted, (900, 600)))
else:
    print('Aucune image de test trouvée pour la visualisation immédiate.')